In [1]:
import cyipopt
import numpy as np

In [2]:
import numpy as np
import cyipopt

class OptimizationProblem:
    """
    Minimize: f(x) = (x[0] - 2)^2 + (x[1] - 1)^2
    
    Subject to:
        g1(x) = x[0]^2 + x[1]^2 - 1 <= 0  (circle constraint)
        g2(x) = x[0] + x[1] - 1 = 0         (linear equality constraint)
    """
    
    def objective(self, x):
        """Objective function to minimize"""
        return (x[0] - 2.0)**2 + (x[1] - 1.0)**2
    
    def gradient(self, x):
        """Gradient of the objective function"""
        return np.array([
            2.0 * (x[0] - 2.0),
            2.0 * (x[1] - 1.0)
        ])
    
    def constraints(self, x):
        """Constraint functions"""
        return np.array([
            x[0]**2 + x[1]**2 - 1.0,  # g1: inequality constraint
            x[0] + x[1] - 1.0          # g2: equality constraint
        ])
    
    def jacobian(self, x):
        """Jacobian of constraints (2 constraints x 2 variables)"""
        return np.array([
            [2.0 * x[0], 2.0 * x[1]],  # gradient of g1
            [1.0, 1.0]                  # gradient of g2
        ]).flatten()  # Return as 1D array
    
    def hessianstructure(self):
        """Define the structure of the Hessian (lower triangular)"""
        # For a 2x2 matrix, lower triangular indices are:
        # (0,0), (1,0), (1,1)
        return np.array([0, 1, 1]), np.array([0, 0, 1])
    
    def hessian(self, x, lagrange, obj_factor):
        """
        Hessian of the Lagrangian (lower triangular values only):
        L = obj_factor * f(x) + lagrange[0] * g1(x) + lagrange[1] * g2(x)
        """
        # Hessian values in lower triangular order: H[0,0], H[1,0], H[1,1]
        H = np.zeros(3)
        
        # H[0,0]: d²L/dx0²
        H[0] = obj_factor * 2.0 + lagrange[0] * 2.0
        
        # H[1,0]: d²L/dx1dx0 (off-diagonal, always 0 for this problem)
        H[1] = 0.0
        
        # H[1,1]: d²L/dx1²
        H[2] = obj_factor * 2.0 + lagrange[0] * 2.0
        
        return H




In [3]:
def solve_optimization():
    """Solve the optimization problem using cyipopt"""
    
    # Problem dimensions
    n_vars = 2  # number of variables
    n_cons = 2  # number of constraints
    
    # Variable bounds (unbounded in this case)
    lb = np.array([-np.inf, -np.inf])
    ub = np.array([np.inf, np.inf])
    
    # Constraint bounds
    # g1 <= 0 means -inf <= g1 <= 0
    # g2 = 0 means 0 <= g2 <= 0
    cl = np.array([-np.inf, 0.0])
    cu = np.array([0.0, 0.0])
    
    # Initial guess
    x0 = np.array([0.5, 0.5])
    
    # Create problem instance
    problem = OptimizationProblem()
    
    # Define the problem for cyipopt
    nlp = cyipopt.Problem(
        n=n_vars,
        m=n_cons,
        problem_obj=problem,
        lb=lb,
        ub=ub,
        cl=cl,
        cu=cu
    )
    
    # Set solver options (optional)
    nlp.add_option('print_level', 5)
    nlp.add_option('tol', 1e-7)
    nlp.add_option('max_iter', 100)
    
    # Solve the problem
    x_opt, info = nlp.solve(x0)
    
    # Print results
    print("\n" + "="*60)
    print("OPTIMIZATION RESULTS")
    print("="*60)
    print(f"Optimal solution: x = [{x_opt[0]:.6f}, {x_opt[1]:.6f}]")
    print(f"Optimal objective value: f(x*) = {problem.objective(x_opt):.6f}")
    print(f"\nConstraint values at solution:")
    print(f"  g1(x*) = {problem.constraints(x_opt)[0]:.6f} (should be <= 0)")
    print(f"  g2(x*) = {problem.constraints(x_opt)[1]:.6f} (should be = 0)")
    print(f"\nSolver status: {info['status_msg']}")
    #print(f"Number of iterations: {info['iter']}")
    print("="*60)
    
    return x_opt, info


In [4]:
x_optimal, solver_info = solve_optimization()


OPTIMIZATION RESULTS
Optimal solution: x = [0.999870, 0.000130]
Optimal objective value: f(x*) = 2.000000

Constraint values at solution:
  g1(x*) = -0.000261 (should be <= 0)
  g2(x*) = 0.000000 (should be = 0)

Solver status: b'Algorithm terminated successfully at a locally optimal point, satisfying the convergence tolerances (can be specified by options).'
